## Tile Operation Graph (TOG)
The compiler writes each kernel's TOG as C++, `tog.cpp`, in the kernel's dump directory, and builds it
into `tog.so`, which TOGSim runs. This notebook compiles a GeMM and a ReLU, each into a directory of
its own under `tog/`, so their TOGs can be opened from the file browser.

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_timing_only.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### GeMM

In [ ]:
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "tog", "gemm")
device = torch.device("npu:0")

input = torch.randn(1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.matmul)
npu_output = opt_fn(input, weight)

### ReLU

In [ ]:
torch._dynamo.reset()
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "tog", "relu")

x = torch.randn(1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.relu)
npu_output = opt_fn(x)

### The TOGs
Open these files from the file browser. `tog_cycles.tsv`, next to each, holds the cycles Gem5
measured for each compute node.

In [ ]:
for path in sorted(glob.glob(os.path.join("tog", "*", "triton_*", "tog.cpp"))):
    print(path)

In [ ]:
print(open(glob.glob(os.path.join("tog", "relu", "triton_*", "tog.cpp"))[0]).read())